In [ ]:
import os
from ast import literal_eval
from time import time

import pyspark.sql.functions as sf
from colabfit.tools.database import batched
from colabfit.tools.schema import config_schema
from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql.types import ArrayType, StringType
from vastdb.session import Session
from tqdm import tqdm

load_dotenv()
endpoint = os.getenv("VAST_DB_ENDPOINT")
access = os.getenv("VAST_DB_ACCESS")
secret = os.getenv("VAST_DB_SECRET")
sess = Session(access=access, secret=secret, endpoint=endpoint)

spark = SparkSession.builder.appName("fix_co_ids").getOrCreate()

In [ ]:
ids = [
    x["id"]
    for x in spark.table("ndb.colabfit.dev.co_wip_combined_rows").select("id").collect()
]
id_batches = batched(ids, 500)
with sess.transaction() as tx:
    for batch in tqdm(id_batches):
        table = tx.bucket("colabfit").schema("dev").table("co_wip")
        row = table.select(predicate=table["id"].isin(batch), internal_row_id=True)
        table.delete(row.read_all())

In [ ]:
# Check whether rows are gone
old_table = spark.table("ndb.colabfit.dev.co_wip")
old_table.filter(sf.col("id") == ids[0]).first()

# Get ds ids
ds_ids = [
    x["dataset_ids"]
    for x in spark.table("ndb.colabfit.dev.co_wip_combined_rows")
    .select("dataset_ids")
    .collect()
]
ds_ids = [literal_eval(x) for x in ds_ids]
ds_ids_flat = list(set([item for sublist in ds_ids for item in sublist]))
spark.table("ndb.colabfit.dev.ds_wip").select("id", "name").filter(
    sf.col("id").isin(ds_ids_flat)
).show(truncate=False)

# Remove the dss from dss table
with sess.transaction() as tx:
    table = tx.bucket("colabfit").schema("dev").table("ds_wip")
    row = table.select(predicate=table["id"].isin(ds_ids_flat), internal_row_id=True)
    table.delete(row.read_all())

In [ ]:
new_rows = spark.table("ndb.colabfit.dev.co_wip_combined_rows")
old_table = "ndb.colabfit.dev.co_wip"
new_rows.write.mode("append").saveAsTable(old_table)

In [ ]:
# now for the POs
from colabfit.tools.database import batched

pos = spark.table("ndb.colabfit.dev.po_wip")
ids = pos.select("id")
ids = ids.groupBy("id")
ids = ids.count().filter("count > 1")
ids.write.mode("errorifexists").saveAsTable("ndb.colabfit.dev.duplicate_po_rows1")

# Working with ids
pos = spark.table("ndb.colabfit.dev.duplicate_po_rows")
new_table_name = "po_row_duplicates_real"
ids = [x["id"] for x in pos.select("id").collect()]
id_batches = batched(ids, 500)
with sess.transaction() as tx:
    for i, batch in tqdm(enumerate(id_batches), total=(len(ids) // 500)):
        table = tx.bucket("colabfit").schema("dev").table("po_wip")
        row = table.select(predicate=table["id"].isin(batch), internal_row_id=False)
        rec_batch = row.read_all()
        if i == 0:
            sch = tx.bucket("colabfit").schema("dev")
            sch.create_table(new_table_name, rec_batch.schema)
        print(f"rec batch len = {len(rec_batch)}")
        table = tx.bucket("colabfit").schema("dev").table(new_table_name)
        table.insert(rec_batch)

# combining the po rows
pos = spark.table("ndb.colabfit.dev.po_row_duplicates_real")
# ids = [x["id"] for x in table.select("id").distinct().collect()]
# id_batches = batched(ids, 500)
combined_rows = pos.groupBy("id").agg(
    sf.sum("multiplicity").alias("multiplicity"),
    *[
        sf.first(col, ignorenulls=True).alias(col)
        for col in pos.columns
        if col not in ["id", "multiplicity"]
    ],
)
combined_rows.write.mode("overwrite").saveAsTable("ndb.colabfit.dev.po_combined_rows")
# check rows
comb_rows = spark.table("ndb.colabfit.dev.po_combined_rows")
comb_rows.first()

ids = [x["id"] for x in comb_rows.select("id").collect()]
assert len(ids) == len(set(ids))
batched_ids = batched(ids, 100)
with sess.transaction() as tx:
    for i, batch in tqdm(enumerate(batched_ids), total=((len(ids) // 500) + 1)):
        table = tx.bucket("colabfit").schema("dev").table("po_wip")
        row = table.select(predicate=table["id"].isin(batch), internal_row_id=True)
        rec_batch = row.read_all()
        print("rec batch len = ", len(rec_batch))
        table.delete(rec_batch)

# append dups to the original table
comb_rows = spark.table("ndb.colabfit.dev.po_combined_rows")
old_table = "ndb.colabfit.dev.po_wip"
comb_rows.write.mode("append").saveAsTable(old_table)

In [2]:
# DSs with POs that were duplicated, need to be re-aggregated

po_ds_ids = set(
    [
        "DS_gn4ti9f2itea_0",
        "DS_5yfdgzb5zhgm_0",
        "DS_k065jfggbq43_0",
        "DS_sng40qq19dak_0",
        "DS_6woak771jubv_0",
        "DS_tz8gsj3xi2g4_0",
        "DS_3h39eqiv9urv_0",
        "DS_82x5bfiiyaij_0",
        "DS_h7gnyidyqcxe_0",
        "DS_6e5ljaqa3cfr_0",
        "DS_4pbhjtu62o2d_0",
        "DS_g0sb0h7usqw7_0",
        "DS_lvixye0ynk1o_0",
        "DS_piuigd7monq9_0",
        "DS_z90lfjg88fzo_0",
        "DS_yq2whjjndyq5_0",
        "DS_qph0akhjv9kv_0",
        "DS_t0ec1irmeo59_0",
        "DS_abagltajle7q_0",
        "DS_krd83tt1g6wd_0",
        "DS_pv1f3dlo5dsc_0",
        "DS_qrkr2xiw1wtp_0",
        "DS_dtjyh96dypuu_0",
        "DS_iie3c31ar46x_0",
        "DS_y8u933mhadjf_0",
        "DS_e94my2wrh074_0",
        "DS_v4e6t79tt5xh_0",
        "DS_720rbshv96l1_0",
        "DS_agiti2oe5bqb_0",
        "DS_vxd0mrt5it19_0",
        "DS_4mjnowmrcqib_0",
        "DS_101uk70asqhq_0",
        "DS_70btumen3361_0",
        "DS_0zgz34a90a6i_0",
        "DS_dc3o40aou2le_0",
        "DS_gc1y80tpyylb_0",
        "DS_c4s38mdirjf7_0",
        "DS_098x6q7kbeat_0",
        "DS_xy48avqcknnk_0",
        "DS_9p3sip4yhiju_0",
        "DS_or3nu4t64mvk_0",
        "DS_mm4npn96qxo1_0",
        "DS_ikbqbyd3dw25_0",
        "DS_k1k8iul6kgm2_0",
        "DS_39v8vn61tlzl_0",
        "DS_caktb6z8yiy7_0",
        "DS_4vdrw3cfi4s7_0",
        "DS_jgaid7espcoc_0",
        "DS_mevqyitwxukc_0",
        "DS_yk3t004l8dpd_0",
        "DS_wyo91w20wlgm_0",
        "DS_4eb78xs9suoo_0",
        "DS_nue14kckbkdh_0",
        "DS_doj9b688juif_0",
        "DS_ngso7es93qnj_0",
        "DS_8k237jm1rd3s_0",
        "DS_qgi1kbtdzwmm_0",
        "DS_xsad0btc0bsn_0",
        "DS_jt0lax9yd15r_0",
        "DS_s818rozdb6x6_0",
        "DS_8y775we7um7w_0",
        "DS_58y020ce6b6j_0",
        "DS_k059wtxqsksu_0",
        "DS_omnl1yy49sdh_0",
        "DS_nb8hcpibz1dt_0",
        "DS_q4h7q8q0fnve_0",
        "DS_tydu7u0h0hhz_0",
        "DS_5lhmgnxhuia3_0",
        "DS_zduxfk2oohzc_0",
        "DS_bnnyjpaqb09i_0",
        "DS_5tegg4uvaixz_0",
        "DS_ofpcyxez6xsc_0",
        "DS_f25kywgtdhks_0",
        "DS_k85kj1kiekip_0",
    ]
)

In [ ]:
# DSs with COs that were duplicated, need to be re-aggregated

co_ds_ids = set(
    [
        "DS_4vdrw3cfi4s7_0",
        "DS_abagltajle7q_0",
        "DS_c4s38mdirjf7_0",
        "DS_h7gnyidyqcxe_0",
        "DS_jcodhqs0atqm_0",
        "DS_krd83tt1g6wd_0",
        "DS_mm4npn96qxo1_0",
        "DS_qrkr2xiw1wtp_0",
        "DS_sng40qq19dak_0",
        "DS_vxd0mrt5it19_0",
        "DS_weq0x6qxqbau_0",
        "DS_zduxfk2oohzc_0",
    ]
)

In [6]:
omat_ds_ids = [
    "DS_vxd0mrt5it19_0",
    "DS_zduxfk2oohzc_0",
    "DS_c4s38mdirjf7_0",
    "DS_8k237jm1rd3s_0",
    "DS_h7gnyidyqcxe_0",
    "DS_qrkr2xiw1wtp_0",
    "DS_sng40qq19dak_0",
    "DS_abagltajle7q_0",
    "DS_mm4npn96qxo1_0",
    "DS_krd83tt1g6wd_0",
    "DS_4vdrw3cfi4s7_0",
]

In [5]:
import pyspark.sql.functions as sf

spark.table("ndb.colabfit.dev.ds_wip").select("id", "name").filter(
    sf.col("id").isin(po_ds_ids)
).show(100, truncate=False)

{'DS_jcodhqs0atqm_0', 'DS_weq0x6qxqbau_0'}

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.types import ArrayType, StringType
import json
from ast import literal_eval
from colabfit.tools.schema import config_schema

edit_cols = [
    "dataset_ids",
    "names",
    "labels",
]


def parse_stringified_array(col):
    return F.udf(lambda x: literal_eval(x) if x else [], ArrayType(StringType()))(
        F.col(col)
    )


table = spark.table("ndb.colabfit.dev.co_omat")

duplicate_ids = table.groupBy("id").count().filter("count > 1").select("id")
dup_rows = table.join(duplicate_ids, on="id", how="inner")

dup_rows = dup_rows.select(
    *[col for col in table.columns if col not in edit_cols],
    parse_stringified_array("dataset_ids").alias("dataset_ids"),
    parse_stringified_array("names").alias("names"),
    parse_stringified_array("labels").alias("labels"),
)
combined_rows = dup_rows.groupBy("id").agg(
    F.flatten(F.collect_set("dataset_ids")).alias("dataset_ids"),
    F.flatten(F.collect_set("names")).alias("names"),
    F.flatten(F.collect_set("labels")).alias("labels"),
    *[
        F.first(col, ignorenulls=True).alias(col)
        for col in table.columns
        if col not in {"id", "dataset_ids", "names", "labels"}
    ],
)

combined_rows = combined_rows.select(
    *[col for col in table.columns if col not in edit_cols],
    *[
        F.udf(lambda x: str(x) if x else None)(F.col(col)).alias(col)
        for col in edit_cols
    ],
)
non_duplicate_rows = table.join(duplicate_ids, on="id", how="left_anti")
final_table = non_duplicate_rows.unionByName(combined_rows).select(
    [col for col in config_schema.fieldNames()]
)

In [ ]:
table = spark.table("ndb.colabfit.dev.co_wip")
dup_ids = table.select("id").groupBy("id").count().filter("count > 1")
dups_rows = table.join(dup_ids, on="id", how="left").select(
    "id", "dataset_ids", "names", "labels"
)
unstring_udf = sf.udf(lambda x: literal_eval(x), ArrayType(StringType()))
dups_rows = dups_rows.select(
    "id", *[unstring_udf(c).alias(c) for c in ["dataset_ids", "names", "labels"]]
)
dups_rows_with_elements_combined_into_one_row = dups_rows.groupBy("id").agg(
    sf.collect_set("dataset_ids").alias("dataset_ids"),
    sf.collect_set("names").alias("names"),
    sf.collect_set("labels").alias("labels"),
)